# 08. Azimuth crop subdivision and worker dispatch

Large crops are split along azimuth before tomographic focusing so each subsection fits in memory and can be processed in parallel. This notebook exercises the real subdivision logic (`TomogramProcessor._divide_crop`, `CropRegion.subdivide_by_azimuth`) and the worker-count resolution (`ParallelConfiguration.resolve_workers`), then visualises the resulting azimuth tiling. No disk IO is involved.

**Modules exercised:** pipelines.processing_pipeline.tomogram (TomogramProcessor._divide_crop), tools.data.regions (CropRegion.subdivide_by_azimuth), configuration.processing_config (ParallelConfiguration)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Import the real components

We import the actual classes and drive their subdivision logic directly. A lightweight logger is constructed because `TomogramProcessor` requires one.

In [ ]:
from configuration.processing_config import (
    ProcessingConfiguration,
    TomogramConfiguration,
    ParallelConfiguration,
    PathConfiguration,
)
from tools.data.regions import CropRegion
from pipelines.processing_pipeline.tomogram import TomogramProcessor
from tools.monitoring.logger import Logger

logger = Logger(log_dir='/tmp/nb08_logs', name='nb08', level='WARNING')
print('imported TomogramProcessor and CropRegion')

## Build a config with a wide crop

We choose a crop wider than `max_crop_azimuth_width` so subdivision actually triggers, mirroring the pipeline where the width is set to one sixteenth of the azimuth extent in `main/pre_process.py`.

In [ ]:
global_crop = CropRegion(azimuth_start=1000, azimuth_end=9000, range_start=500, range_end=4000)
max_width   = (global_crop.azimuth_end - global_crop.azimuth_start) // 8

tomo_cfg = TomogramConfiguration(max_crop_azimuth_width=max_width)
config   = ProcessingConfiguration(
    crop          = global_crop,
    input_configs = tomo_cfg,
    parallel      = ParallelConfiguration(pyrat_threads=4),
    paths         = PathConfiguration(main_directory=Path('/tmp/nb08_run')),
)
print('crop azimuth extent:', global_crop.azimuth_size)
print('max_crop_azimuth_width:', max_width)

## Run the real subdivision

`_divide_crop` returns a list of `(az_start, az_end, rg_start, rg_end)` tuples. The number of workers is then resolved from the subsection count and the parallel configuration.

In [ ]:
processor   = TomogramProcessor(config, logger)
subsections = processor._divide_crop(tomo_cfg)
workers     = config.parallel.resolve_workers(len(subsections))

print('number of subsections:', len(subsections))
for i, s in enumerate(subsections):
    print(f'  subsection {i:02d}: azimuth [{s[0]}, {s[1]}) width={s[1]-s[0]} range [{s[2]}, {s[3]})')
print('resolved workers:', workers, 'on', config.parallel.available_cores(), 'cores')

## Visualise the azimuth tiling

Each subsection covers a contiguous azimuth band of width at most `max_crop_azimuth_width`; the last band may be narrower. The tiles should fully cover the crop without gaps or overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
colors = plt.cm.tab20(np.linspace(0, 1, len(subsections)))
for i, (s, c) in enumerate(zip(subsections, colors)):
    ax.barh(0, s[1] - s[0], left=s[0], color=c, edgecolor='k', height=0.6)
    ax.text((s[0] + s[1]) / 2, 0, str(i), ha='center', va='center', fontsize=8)
ax.set_xlim(global_crop.azimuth_start, global_crop.azimuth_end)
ax.set_yticks([])
ax.set_xlabel('azimuth sample')
ax.set_title(f'{len(subsections)} azimuth subsections (width <= {max_width})')
fig.tight_layout()
plt.show()

## Worker count across hypothetical core budgets

`resolve_workers` caps the worker count at the subsection count and at `cores // pyrat_threads`. We sweep `tomogram_workers` to show the cap behaviour.

In [ ]:
requested = [None, 1, 2, 4, 8, 100]
resolved  = []
for r in requested:
    pc = ParallelConfiguration(tomogram_workers=r, pyrat_threads=4)
    resolved.append(pc.resolve_workers(len(subsections)))
labels = ['auto' if r is None else str(r) for r in requested]

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(labels, resolved, color='0.5', edgecolor='k')
ax.axhline(len(subsections), color='r', ls='--', lw=0.9, label=f'subsection cap = {len(subsections)}')
ax.set_xlabel('requested tomogram_workers'); ax.set_ylabel('resolved workers')
ax.set_title('resolve_workers behaviour')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The tiling bar should show contiguous, non-overlapping azimuth bands spanning the full crop, the last one possibly shorter. The worker-count bars should never exceed the subsection cap (red line), confirming `resolve_workers` clamps requests to the available work and core budget.